In [53]:
# Imports
import os
import pandas as pd
import numpy as np

import plotly.io as pio
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Renderer
pio.renderers.default = "notebook"

# Check file names in data/raw
print(os.listdir("../data/raw"))

['gencode.v36.annotation.gtf.gene.probemap', 'TCGA-BRCA.clinical.tsv.gz', 'TCGA-BRCA.star_counts.tsv.gz', 'TCGA-BRCA.survival.tsv.gz']


# Gene Expression Matrix

In [31]:
raw_expr = pd.read_csv("../data/raw/TCGA-BRCA.star_counts.tsv.gz", sep="\t", index_col=0)

print("Shape:", raw_expr.shape)
display(raw_expr.head())

Shape: (60660, 1226)


,TCGA-D8-A146-01A,TCGA-AQ-A0Y5-01A,TCGA-C8-A274-01A,TCGA-BH-A0BD-01A,TCGA-B6-A1KC-01B,TCGA-AC-A62V-01A,TCGA-AO-A0J5-01A,TCGA-BH-A0B1-01A,TCGA-A2-A0YM-01A,TCGA-AO-A03N-01B,...,TCGA-E2-A1IG-01A,TCGA-E9-A1NA-01A,TCGA-D8-A1JP-01A,TCGA-AR-A252-01A,TCGA-D8-A1XL-01A,TCGA-BH-A0EI-01A,TCGA-E2-A1IO-01A,TCGA-E2-A15R-01A,TCGA-B6-A0IP-01A,TCGA-A1-A0SN-01A
Ensembl_ID,,,,,,,,,,,,,,,,,,,,,
ENSG00000000003.15,11.737670,9.781360,13.122504,11.016808,11.000000,9.614710,11.092096,12.103616,12.417325,10.430453,...,10.747354,10.360847,10.675957,11.337064,11.697836,12.339850,12.131857,8.974415,13.326289,9.710806
ENSG00000000005.6,7.721099,3.321928,0.000000,6.686501,3.807355,4.087463,5.392317,3.000000,3.321928,0.000000,...,1.584963,2.000000,1.000000,8.164907,0.000000,2.807355,3.906891,3.700440,3.807355,1.000000
ENSG00000000419.13,11.042343,11.357552,11.506308,10.801708,11.074141,11.107217,10.318543,11.160502,11.072803,11.088126,...,10.429407,10.811375,10.835261,10.693487,11.922213,11.221587,10.913637,11.659550,10.962173,12.416270
ENSG00000000457.14,11.036860,10.754888,12.218260,11.190442,10.857981,8.539159,11.527966,10.635718,9.945444,9.499846,...,10.366322,10.558421,11.262682,10.751544,10.607330,10.163650,10.575539,11.929258,11.403012,10.870365
ENSG00000000460.17,9.131857,8.721099,10.973697,10.761551,9.550747,8.550747,9.177420,9.727920,10.504819,8.584963,...,8.661778,9.000000,9.794416,9.231221,9.782998,9.252665,8.632995,10.562242,10.060696,9.722808


**Dataset** : *60661* identifiers (rows) x *1226* samples (columns)

**Identifiers** : transcribed genomic locus on the human reference genome

ex : ENSG00000141510.18 (which maps to TP53) 
- ENSG = Ensembl Gene identifier.
-  00000141510 = Unique 11-digit catalog number for that exact DNA region.
- .18 = Annotation version number (how many times exon/intron boundaries for that gene model have been refined).

**Samples** : distinct biological tissue biopsy (every single one of the 1,226 columns is a separate biological tissue sample).

ex : TCGA - BH - A0B3 - 01A
- TCGA : The Cancer Genome Atlas project
- BH : Tissue Source Site (e.g. BH = University of Pittsburgh) --> useful later for checking batch effects
- A0B3 : Participant ID
- 01 : Sample Type (what biological tissue)
- A : vial

*Why 1226 columns for 1095 patients ? some patients donated more than one type of tissue !*
- -01 : Primary Solid Tumor
- -11 : Solid Tissue Normal
- -06 : Metastatic

**Cells** : Each cell value in the matrix is $\log_2(\text{STAR raw read count} + 1)$ — the log-transformed number of sequencing reads that mapped to that gene's exons in that tissue sample.


# Gene Mapping

In [32]:
probe_map = pd.read_csv("../data/raw/gencode.v36.annotation.gtf.gene.probemap", sep="\t")

expr = raw_expr.copy()
id_to_gene = probe_map.set_index("id")["gene"]
expr.insert(0, "gene", expr.index.map(id_to_gene))

print("Unmapped IDs:", expr["gene"].isna().sum())
print("Duplicated symbols:", expr["gene"].duplicated().sum())

display(expr.head())

Unmapped IDs: 0
Duplicated symbols: 1233


,gene,TCGA-D8-A146-01A,TCGA-AQ-A0Y5-01A,TCGA-C8-A274-01A,TCGA-BH-A0BD-01A,TCGA-B6-A1KC-01B,TCGA-AC-A62V-01A,TCGA-AO-A0J5-01A,TCGA-BH-A0B1-01A,TCGA-A2-A0YM-01A,...,TCGA-E2-A1IG-01A,TCGA-E9-A1NA-01A,TCGA-D8-A1JP-01A,TCGA-AR-A252-01A,TCGA-D8-A1XL-01A,TCGA-BH-A0EI-01A,TCGA-E2-A1IO-01A,TCGA-E2-A15R-01A,TCGA-B6-A0IP-01A,TCGA-A1-A0SN-01A
Ensembl_ID,,,,,,,,,,,,,,,,,,,,,
ENSG00000000003.15,TSPAN6,11.737670,9.781360,13.122504,11.016808,11.000000,9.614710,11.092096,12.103616,12.417325,...,10.747354,10.360847,10.675957,11.337064,11.697836,12.339850,12.131857,8.974415,13.326289,9.710806
ENSG00000000005.6,TNMD,7.721099,3.321928,0.000000,6.686501,3.807355,4.087463,5.392317,3.000000,3.321928,...,1.584963,2.000000,1.000000,8.164907,0.000000,2.807355,3.906891,3.700440,3.807355,1.000000
ENSG00000000419.13,DPM1,11.042343,11.357552,11.506308,10.801708,11.074141,11.107217,10.318543,11.160502,11.072803,...,10.429407,10.811375,10.835261,10.693487,11.922213,11.221587,10.913637,11.659550,10.962173,12.416270
ENSG00000000457.14,SCYL3,11.036860,10.754888,12.218260,11.190442,10.857981,8.539159,11.527966,10.635718,9.945444,...,10.366322,10.558421,11.262682,10.751544,10.607330,10.163650,10.575539,11.929258,11.403012,10.870365
ENSG00000000460.17,C1orf112,9.131857,8.721099,10.973697,10.761551,9.550747,8.550747,9.177420,9.727920,10.504819,...,8.661778,9.000000,9.794416,9.231221,9.782998,9.252665,8.632995,10.562242,10.060696,9.722808


Inserted a gene column for easier interpretation.

In [33]:
sample_cols = expr.columns[1:]
sample_types = sample_cols.str.split("-").str[3].str[:2]

print("Unique patients (12-char):", sample_cols.str[:12].nunique())
print(sample_types.value_counts())

Unique patients (12-char): 1095
01    1106
11     113
06       7
Name: count, dtype: int64


# Clinical file

In [34]:
clin = pd.read_csv("../data/raw/TCGA-BRCA.clinical.tsv.gz", sep="\t")

print("Shape:", clin.shape)
display(clin.head())
display(clin.isna().sum().to_frame("missing_count"))

Shape: (1255, 85)


,sample,id,disease_type,case_id,submitter_id,primary_site,alcohol_history.exposures,race.demographic,gender.demographic,ethnicity.demographic,...,days_to_collection.samples,initial_weight.samples,preservation_method.samples,pathology_report_uuid.samples,oct_embedded.samples,specimen_type.samples,days_to_sample_procurement.samples,is_ffpe.samples,tissue_type.samples,annotations.samples
0,TCGA-BH-A0W3-01A,3c612e12-6de8-44fa-a095-805c45474821,Ductal and Lobular Neoplasms,3c612e12-6de8-44fa-a095-805c45474821,TCGA-BH-A0W3,Breast,Not Reported,white,female,not hispanic or latino,...,85.0,120.0,OCT,801A4E2F-E26E-424F-BF42-CD0D9CD62BCE,True,Solid Tissue,NaN,False,Tumor,NaN
1,TCGA-AR-A24V-01A,3cb06c7a-f2a8-448b-91a8-dd201bbf2ddd,Ductal and Lobular Neoplasms,3cb06c7a-f2a8-448b-91a8-dd201bbf2ddd,TCGA-AR-A24V,Breast,Not Reported,white,female,not hispanic or latino,...,1720.0,400.0,OCT,468CD293-C9F7-43C6-A40A-18FCDD22F6AA,True,Solid Tissue,NaN,False,Tumor,NaN
2,TCGA-E9-A1NE-01A,3d676bba-154b-4d22-ab59-d4d4da051b94,Ductal and Lobular Neoplasms,3d676bba-154b-4d22-ab59-d4d4da051b94,TCGA-E9-A1NE,Breast,Not Reported,white,female,not hispanic or latino,...,31.0,280.0,OCT,CF6E29A2-FAE6-45BB-B625-33877887A89E,True,Solid Tissue,NaN,False,Tumor,NaN
3,TCGA-E9-A1NE-11A,3d676bba-154b-4d22-ab59-d4d4da051b94,Ductal and Lobular Neoplasms,3d676bba-154b-4d22-ab59-d4d4da051b94,TCGA-E9-A1NE,Breast,Not Reported,white,female,not hispanic or latino,...,31.0,830.0,OCT,NaN,True,Solid Tissue,NaN,False,Normal,NaN
4,TCGA-AC-A8OQ-01A,dfaabd03-2d40-4422-b210-caf112ff4229,Ductal and Lobular Neoplasms,dfaabd03-2d40-4422-b210-caf112ff4229,TCGA-AC-A8OQ,Breast,Not Reported,black or african american,female,not hispanic or latino,...,742.0,100.0,Unknown,FFA6F9F3-71C1-4AF9-B9F7-0466550EBC90,False,Solid Tissue,NaN,False,Tumor,NaN


,missing_count
sample,0
id,0
disease_type,0
case_id,0
submitter_id,0
...,...
specimen_type.samples,0
days_to_sample_procurement.samples,1251
is_ffpe.samples,0
tissue_type.samples,0


# Survival file

In [35]:
surv = pd.read_csv("../data/raw/TCGA-BRCA.survival.tsv.gz", sep="\t")

print("Shape:", surv.shape) # Table dimensions
display(surv.head()) 
display(surv.isna().sum().to_frame("missing_count")) # No missing values
print(surv["OS"].value_counts(dropna=False))
display(surv.describe())

Shape: (1232, 4)


,sample,OS.time,OS,_PATIENT
0,TCGA-C8-A275-01A,1.0,0,TCGA-C8-A275
1,TCGA-AC-A7VC-01A,1.0,0,TCGA-AC-A7VC
2,TCGA-BH-A1F8-01A,1.0,1,TCGA-BH-A1F8
3,TCGA-BH-A1F8-11B,1.0,1,TCGA-BH-A1F8
4,TCGA-PL-A8LX-01A,5.0,0,TCGA-PL-A8LX


,missing_count
sample,0
OS.time,0
OS,0
_PATIENT,0


OS
0    1031
1     201
Name: count, dtype: int64


,OS.time,OS
count,1232.00000,1232.000000
mean,1273.17776,0.163149
std,1154.55418,0.369652
min,1.00000,0.000000
25%,486.50000,0.000000
50%,922.00000,0.000000
75%,1734.50000,0.000000
max,8605.00000,1.000000


**OS** : Overall Survival

- 1 : Patient died (201)
- 0 : Patient was still alive at their last recorded hospital visit (survived at least until OS.time

**OS.time** : Overall Survival time

- Date of death (when OS = 1) : number of days from diagnosis to death
- Date of last contact (when OS = 0)


**Methodological Warning — Survival Endpoint Traps**
> Per project guidelines, if we select survival (`OS`, `OS.time`) as our prediction target, we must get instructor approval and use censoring-aware methods. Standard supervised formulations are invalid here:
>
> * **Trap 1 — Modeling `OS.time` with standard regression (MSE / MAE):** Right-censored patients (`OS = 0`) were still alive at their last recorded visit, meaning their true survival time is *at least* `OS.time`, not equal to it. Treating `OS.time` as a fully observed target falsely penalizes the model as if censored patients died on their last follow-up date.
> * **Trap 2 — Modeling `OS` with standard binary classification (0 vs. 1):** Reducing survival to a binary label ignores *when* events or censoring occurred—treating a patient lost to follow-up after 30 days (`OS = 0`) identically to one confirmed alive after 10 years (`OS = 0`).
>
> **Valid alternative:** If we use survival, we must train a censoring-aware model (e.g., a Cox proportional hazards MLP) and evaluate it with a survival metric such as the **Concordance Index (C-index)**.

# 01. Exploratory Data Analysis - Gene Expression 

### Sample-Level Expression Distributions (50 Random Biopsies)

In [46]:
# Exclude non-numeric annotation columns (e.g., mapped HGNC gene symbols)
X = expr.select_dtypes(include="number")

# Drop unexpressed/low-count loci so zero-inflation (~2/3 of GENCODE features)
# does not collapse sample medians and IQRs onto 0
X_expressed = X[X.mean(axis=1) > 1]

# Reproducibly subsample 50 distinct biopsies to keep sample-level median shifts
# visually legible without overloading browser rendering across all 1,226 columns
rng = np.random.default_rng(42)
sampled_cols = rng.choice(X_expressed.columns, size=50, replace=False)

fig = go.Figure()
for col in sampled_cols:
    # Distinguish adjacent normal tissue (-11) from tumor biopsies (-01/-06)
    # to separate biological tissue shifts from technical library-size offsets
    is_normal = col.split("-")[3].startswith("11")
    fig.add_trace(go.Box(
        y=X_expressed[col],
        name=col[:15],
        marker_color="#2ca02c" if is_normal else "#1f77b4",
        # Suppress individual outlier dots to keep rendering fast and focus on medians/IQRs
        boxpoints=False,
        showlegend=False
    ))

fig.update_layout(
    title="Expression Distribution Across 50 Random Samples (Mean > 1)",
    yaxis_title="log2(count + 1)",
    xaxis_tickangle=-45,
    height=500
)
fig.show()

* **Unnormalized sequencing depth:** The middle lines (medians, ranging from `~4.8` to `6.8`) and vertical box positions shift up and down across biopsies—for example, `TCGA-B6-A0RE-01` and `TCGA-AR-A0TP-01` sit noticeably lower than `TCGA-E2-A11E-01` or `TCGA-B6-A401-01`. Because biopsies were sequenced at different depths, we must normalize library sizes first so PCA captures true tumor biology rather than how deeply each sample was sequenced.
* **Healthy expression spread across all samples:** After filtering out silent genes ($\text{mean} \le 1$), all 50 biopsies show comparable box heights (IQR from `~2` to `10.5`) and upper limits (`~18` to `22`). This confirms that the $\log_2(\text{count}+1)$ scale prevents extreme high-count genes from squashing the distribution and that none of the sampled biopsies suffered technical sequencing failure (which would appear as a box collapsed near `0`).
* **Higher, more consistent medians in normal tissue:** The four adjacent normal breast samples (`-11`, in green: `TCGA-C8-A12M-11`, `TCGA-BH-A18M-11`, `TCGA-BH-A0DG-11`, and `TCGA-A7-A0CE-11`) sit consistently higher (medians around `6.2`–`6.7`) and vary less from sample to sample than primary tumors (`-01`, in blue). Normal breast tissue maintains a stable baseline expression profile, whereas gene expression varies much more widely across tumors.

## Cohort sequencing depth

In [44]:
raw_counts = np.power(2, X) - 1
lib_sizes = raw_counts.sum(axis=0)

print("Total mapped reads per sample:")
print(lib_sizes.describe().apply(lambda v: f"{v:,.0f}"))
print(f"Max / Min library size ratio: {lib_sizes.max() / lib_sizes.min():.2f}x")

Total mapped reads per sample:
count          1,226
mean      57,993,787
std       14,561,577
min       19,225,117
25%       47,717,104
50%       57,637,384
75%       68,271,050
max      114,015,705
dtype: str
Max / Min library size ratio: 5.93x


**Sequencing depth (library size):** The total number of RNA-seq reads mapped across all 60,660 genomic loci for a single biopsy ($\sum \text{raw counts}$ down a sample column). Variation in sequencing depth across samples (`19.2M` to `114.0M` reads here, a `5.93x` ratio) reflects technical differences in library preparation and sequencer loading rather than biological differences in total cellular RNA.
* **Confirmation of unnormalized raw counts:** Reversing the $\log_2(\text{count} + 1)$ transform gives a median of **57.6 million** mapped reads per biopsy (with half of all samples falling between **47.7M** and **68.3M**). This matches standard bulk RNA-seq sequencing depth and confirms the matrix stores raw read counts rather than pre-normalized values.
* **Substantial variation in sequencing depth (`5.93x` ratio):** Total mapped reads range from **19.2 million** in the shallowest sample to **114.0 million** in the deepest sample—a nearly 6-fold technical spread across the cohort.
* **Why library-size normalization is required before PCA:** Because total read counts vary by almost 6× across biopsies, the same gene can show roughly 6× higher raw counts in a deeply sequenced sample purely for technical reasons. Scaling each sample by its total library size (e.g., $\log_2(\text{CPM} + 1)$) removes this depth bias so PCA captures real biological differences.

## Cohort-Wide Quality & Expression Diagnostics

In [49]:
# Group sample barcodes by tissue type (-01 Tumor, -11 Normal, -06 Metastatic)
# to check for systematic sequencing depth bias between biological groups
sample_meta = pd.DataFrame({
    "sample": X.columns,
    "lib_size_millions": raw_counts.sum(axis=0).values / 1e6,
    "tissue_code": X.columns.str.split("-").str[3].str[:2]
})
sample_meta["tissue"] = sample_meta["tissue_code"].map({
    "01": "Primary Tumor (-01)",
    "11": "Solid Tissue Normal (-11)",
    "06": "Metastatic (-06)"
}).fillna("Other")

# Compute gene-level mean expression across the cohort to separate
# silent background loci (~0) from actively expressed genes
mean_gene_expr = X.mean(axis=1)

fig = make_subplots(
    rows=1, cols=2,
    horizontal_spacing=0.14,
    subplot_titles=(
        "Library Size Distribution (n = 1,226)",
        "Mean Gene Expression (60,660 Loci)"
    )
)

# Panel 1: Overlaid histograms to verify whether tumor and normal biopsies
# share the same sequencing depth range
for tissue, color in [("Primary Tumor (-01)", "#1f77b4"), ("Solid Tissue Normal (-11)", "#2ca02c")]:
    subset = sample_meta[sample_meta["tissue"] == tissue]
    fig.add_trace(
        go.Histogram(x=subset["lib_size_millions"], name=tissue, marker_color=color, opacity=0.7, nbinsx=40),
        row=1, col=1
    )

# Panel 2: Bimodal density check to justify low-expression filtering before PCA
fig.add_trace(
    go.Histogram(x=mean_gene_expr, name="All 60,660 Loci", marker_color="#636efa", nbinsx=60, showlegend=False),
    row=1, col=2
)

fig.update_layout(
    barmode="overlay",
    height=440,
    margin=dict(t=100, b=60, l=60, r=40),
    title_text="Cohort-Wide Quality & Expression Diagnostics",
    legend=dict(orientation="h", yanchor="bottom", y=1.08, xanchor="right", x=1)
)
fig.update_annotations(font_size=13)
fig.update_xaxes(title_text="Total Mapped Reads (Millions)", row=1, col=1)
fig.update_yaxes(title_text="Sample Count", row=1, col=1)
fig.update_xaxes(title_text="Mean log2(count + 1) Across Cohort", row=1, col=2)
fig.update_yaxes(title_text="Number of Genomic Loci", row=1, col=2)
fig.show()

* **No sequencing depth bias between tumor and normal samples:** In the left panel, both Primary Tumors (`-01`, blue) and Solid Tissue Normals (`-11`, green) follow the same bell-shaped distribution centered around **50–65 million** mapped reads, with no failed libraries near `0`. This confirms that normal biopsies were not sequenced deeper than tumors, so *the higher medians observed earlier in normal tissue may reflect real biology rather than a tissue-specific depth artifact*.
* **Clear split between silent loci and active genes:** In the right panel, gene expression across the 60,660 loci forms two distinct groups: a massive spike near `0` (over **20,000** unexpressed or barely detected loci) and an active transcriptome peak centered around `~10–12` $\log_2(\text{count}+1)$.
* **Justification for low-expression filtering:** Because roughly half to two-thirds of the GENCODE catalog sits near `0` across the cohort, filtering out silent loci before PCA and MLP training removes pure background noise while keeping the active breast transcriptome intact.

## Library Size Normalization

5.93x library-size spread, misaligned sample medians : need to scale every sample to **Counts Per Million (CPM)** and return to log2 scale.

In [50]:
# Normalize raw counts to Counts Per Million (CPM) in linear space so every biopsy
# sums to 1,000,000 mapped reads, removing the 5.93x technical sequencing depth bias
cpm = raw_counts.div(lib_sizes, axis=1) * 1e6

# Re-apply log2(x + 1) transformation after depth scaling to compress extreme
# high-expression spikes and stabilize variance prior to PCA
X_norm = np.log2(cpm + 1)

print("Normalized matrix shape (genes x samples):", X_norm.shape)
print("Post-normalization CPM sum per sample (min / max):", f"{cpm.sum(axis=0).min():,.0f} / {cpm.sum(axis=0).max():,.0f}")

# Re-plot the exact same 50 random biopsies after CPM normalization (filtering silent loci
# with mean <= 1) to visually verify that sample medians and IQRs are now aligned
X_norm_expressed = X_norm[X_norm.mean(axis=1) > 1]

fig = go.Figure()
for col in sampled_cols:
    # Keep the same color coding (-11 normal vs. -01/-06 tumor) to confirm that
    # biological tissue differences remain intact after removing technical depth shifts
    is_normal = col.split("-")[3].startswith("11")
    fig.add_trace(go.Box(
        y=X_norm_expressed[col],
        name=col[:15],
        marker_color="#2ca02c" if is_normal else "#1f77b4",
        boxpoints=False,
        showlegend=False
    ))

fig.update_layout(
    title="Post-Normalization Expression Across the Same 50 Samples (log2(CPM + 1), Mean > 1)",
    yaxis_title="log2(CPM + 1)",
    xaxis_tickangle=-45,
    height=500
)
fig.show()

Normalized matrix shape (genes x samples): (60660, 1226)
Post-normalization CPM sum per sample (min / max): 1,000,000 / 1,000,000


* **Technical depth shifts removed:** Scaling each biopsy to Counts Per Million ($\log_2(\text{CPM} + 1)$) aligns the 50 sample medians into a tight band around **`4.0` to `4.6`** and brings interquartile ranges into a consistent window (**`~2.2` to `5.8`**). Low-depth samples that previously dipped downward (such as `TCGA-AR-A0TP-01`) are now aligned with the rest of the cohort.
* **Minor residual median differences reflect RNA composition:** Because CPM normalizes by total mapped reads rather than forcing identical medians, slight sample-to-sample fluctuations remain when a few highly abundant tumor genes take up a larger fraction of a sample's sequencing reads.
* **Ready for feature filtering and PCA:** With the 5.93× sequencing depth bias removed, variance across biopsies now reflects relative gene expression rather than total read count.

## Low Expression Filtering, HVG selection

In [55]:
# Retain loci with log2(CPM + 1) > 1 in at least 20% of biopsies to eliminate
# zero-inflated background noise while preserving subtype-specific markers
expressed_mask = (X_norm > 1).mean(axis=1) >= 0.20
X_filtered = X_norm.loc[expressed_mask]

# Compute mean and variance across all 60,660 loci so the complete two-stage
# filtering funnel (noise -> expressed -> HVG) can be audited in one view
all_means = X_norm.mean(axis=1)
all_vars = X_norm.var(axis=1)

# Select the top 2,000 most variable expressed genes so PCA captures major
# biological variation rather than flat housekeeping genes after unit-variance scaling
top_n = 2000
hvg_ids = all_vars.loc[expressed_mask].nlargest(top_n).index
X_hvg = X_filtered.loc[hvg_ids]

print(f"Loci passing expression filter (>=20% samples > 1): {X_filtered.shape[0]:,} / {X_norm.shape[0]:,}")
print(f"Top HVGs selected for exploratory PCA: {X_hvg.shape[0]:,}")

# Biological sanity check: verify that top-variance genes map to known breast
# markers rather than technical or alignment artifacts
top10_summary = pd.DataFrame({
    "symbol": expr.loc[hvg_ids[:10], "gene"],
    "mean_log2_cpm": all_means.loc[hvg_ids[:10]].round(2),
    "variance": all_vars.loc[hvg_ids[:10]].round(2)
})
display(top10_summary)

# Partition the transcriptome into three mutually exclusive tiers for visual audit
hvg_mask = X_norm.index.isin(hvg_ids)
stable_mask = expressed_mask & (~hvg_mask)
noise_mask = ~expressed_mask
var_cutoff = all_vars.loc[hvg_ids].min()

fig = go.Figure()

# Layer 1 (back): Unexpressed/low-count loci filtered out before variance ranking
fig.add_trace(go.Scattergl(
    x=all_means[noise_mask],
    y=all_vars[noise_mask],
    mode="markers",
    name=f"Low-Expression Noise ({noise_mask.sum():,})",
    marker=dict(color="#d3d3d3", size=3, opacity=0.35)
))

# Layer 2 (middle): Expressed but low-variance housekeeping genes excluded from PCA
fig.add_trace(go.Scattergl(
    x=all_means[stable_mask],
    y=all_vars[stable_mask],
    mode="markers",
    name=f"Stable Expressed Genes ({stable_mask.sum():,})",
    marker=dict(color="#636efa", size=4, opacity=0.45)
))

# Layer 3 (front): Top HVGs carrying the primary biological signal across biopsies
fig.add_trace(go.Scattergl(
    x=all_means[hvg_mask],
    y=all_vars[hvg_mask],
    mode="markers",
    name=f"Top {top_n:,} HVGs ({hvg_mask.sum():,})",
    marker=dict(color="#d62728", size=4, opacity=0.7)
))

# Mark the implied variance boundary separating stable expressed genes from HVGs
fig.add_hline(
    y=var_cutoff,
    line_dash="dash",
    line_color="#333333",
    annotation_text=f"HVG cutoff (var = {var_cutoff:.2f})",
    annotation_position="top right"
)

fig.update_layout(
    title="Mean–Variance Funnel Across All 60,660 Genomic Loci (log2(CPM + 1))",
    xaxis_title="Mean Expression (log2(CPM + 1))",
    yaxis_title="Variance Across Samples",
    height=460,
    legend=dict(itemsizing="constant")
)
fig.show()

Loci passing expression filter (>=20% samples > 1): 17,196 / 60,660
Top HVGs selected for exploratory PCA: 2,000


,symbol,mean_log2_cpm,variance
Ensembl_ID,,,
ENSG00000110484.7,SCGB2A2,5.93,17.10
ENSG00000153002.12,CPB1,3.59,15.41
ENSG00000166509.12,CLEC3A,2.55,12.58
ENSG00000172551.11,MUCL1,4.12,12.37
ENSG00000159763.4,PIP,4.87,11.88
ENSG00000124935.4,SCGB1D2,4.09,11.74
ENSG00000160182.3,TFF1,4.75,10.97
ENSG00000012223.13,LTF,7.05,10.96
ENSG00000256612.7,CYP2B7P,4.21,10.35


* **Three distinct transcriptional tiers:** Partitioning the full GENCODE catalog separates the dataset into **43,464** low-expression noise loci (light grey, 71.7%), **15,196** stable expressed genes (blue, 25.1%), and **2,000** highly variable genes (red, 3.3%) above the implied variance cutoff of **`1.54`**.
* **Rare-subgroup outliers cleanly filtered out:** Nearly all discarded low-expression loci sit tightly compressed near the bottom-left corner ($\text{mean} < 1$, $\text{variance} < 1.54$). Only a tiny handful of grey points extend above `1.54` (around $\text{mean} \approx 0.5\text{–}0.8$, $\text{variance} \le 4$), representing genes active in fewer than 20% of biopsies (such as male-specific Y-chromosome transcripts in ~1% of the cohort). Filtering them out prevents rare sample outliers from distorting global PCA directions.
* **Housekeeping genes separated from biological drivers:** High-abundance, low-variance genes ($\text{mean} > 8$, $\text{variance} < 1.54$, in blue) are excluded before unit-variance scaling (`StandardScaler`), preventing constitutively active housekeeping genes from diluting tumor heterogeneity in PCA.

## Transpose, standardize, fit exploratory PCA

In [56]:
# Transpose from (genes x samples) to (samples x genes) so each biopsy is treated
# as an observation and each HVG as a feature per scikit-learn's convention
X_samples = X_hvg.T

# Standardize each gene to zero mean and unit variance so a handful of extreme-variance
# genes (e.g., SCGB2A2 at var=17.1) do not dominate PCA over coordinated pathways
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_samples)

# Fit 20 principal components to inspect both the variance decay curve (scree plot)
# and the leading biological axes across the cohort
pca = PCA(n_components=20, random_state=42)
pcs = pca.fit_transform(X_scaled)

# Combine PC coordinates with sample metadata to audit biological separation
# against technical confounders (library size and hospital Tissue Source Site)
pca_df = sample_meta.copy()
pca_df["PC1"] = pcs[:, 0]
pca_df["PC2"] = pcs[:, 1]
pca_df["PC3"] = pcs[:, 2]
pca_df["tss"] = pca_df["sample"].str.split("-").str[1]

# Quantify residual correlation between leading PCs and sequencing depth
# to verify that log2(CPM + 1) normalization removed library-size bias
r_pc1_depth = np.corrcoef(pca_df["PC1"], pca_df["lib_size_millions"])[0, 1]
r_pc2_depth = np.corrcoef(pca_df["PC2"], pca_df["lib_size_millions"])[0, 1]

print(f"Variance explained — PC1: {pca.explained_variance_ratio_[0]:.1%}, PC2: {pca.explained_variance_ratio_[1]:.1%}, Top 20 cumulative: {pca.explained_variance_ratio_.sum():.1%}")
print(f"Pearson r with library size — PC1: {r_pc1_depth:.3f}, PC2: {r_pc2_depth:.3f}")

fig = make_subplots(
    rows=1, cols=2,
    horizontal_spacing=0.12,
    subplot_titles=(
        "Scree Plot — Variance Explained (Top 20 PCs)",
        f"PCA (Top {top_n:,} HVGs) by Tissue Type"
    )
)

# Panel 1: Scree plot to assess how rapidly variance concentrates in the leading components
fig.add_trace(
    go.Bar(
        x=[f"PC{i}" for i in range(1, 21)],
        y=pca.explained_variance_ratio_ * 100,
        marker_color="#636efa",
        showlegend=False
    ),
    row=1, col=1
)

# Panel 2: Project samples onto PC1 vs PC2 colored by tissue type (-01, -11, -06)
# to check whether malignant vs. normal biology drives primary cohort structure
palette = {
    "Primary Tumor (-01)": "#1f77b4",
    "Solid Tissue Normal (-11)": "#2ca02c",
    "Metastatic (-06)": "#d62728"
}
for tissue, color in palette.items():
    sub = pca_df[pca_df["tissue"] == tissue]
    fig.add_trace(
        go.Scatter(
            x=sub["PC1"],
            y=sub["PC2"],
            mode="markers",
            name=tissue,
            text=sub["sample"] + " | TSS: " + sub["tss"] + " | Depth: " + sub["lib_size_millions"].round(1).astype(str) + "M",
            hoverinfo="text",
            marker=dict(color=color, size=6, opacity=0.75)
        ),
        row=1, col=2
    )

fig.update_layout(
    height=460,
    margin=dict(t=100, b=60, l=60, r=40),
    title_text="Exploratory PCA on Library-Size Normalized TCGA-BRCA Expression",
    legend=dict(orientation="h", yanchor="bottom", y=1.08, xanchor="right", x=1)
)
fig.update_annotations(font_size=13)
fig.update_xaxes(title_text="Principal Component", row=1, col=1)
fig.update_yaxes(title_text="Explained Variance (%)", row=1, col=1)
fig.update_xaxes(title_text=f"PC1 ({pca.explained_variance_ratio_[0]:.1%})", row=1, col=2)
fig.update_yaxes(title_text=f"PC2 ({pca.explained_variance_ratio_[1]:.1%})", row=1, col=2)
fig.show()

Variance explained — PC1: 14.7%, PC2: 12.9%, Top 20 cumulative: 58.5%
Pearson r with library size — PC1: -0.040, PC2: 0.019


* **Library-size bias completely removed:** Pearson correlation between sequencing depth and the leading axes is near zero (**$r = -0.040$** for PC1, **$r = 0.019$** for PC2), confirming that $\log_2(\text{CPM} + 1)$ normalization eliminated the 5.93× read-depth artifact.
* **Strong biological signal concentration:** PC1 (**14.7%**) and PC2 (**12.9%**) together capture **27.6%** of total variance across 2,000 standardized genes, and the top 20 PCs (1% of feature dimensions) retain **58.5%**, with a clear scree elbow around PC3–PC5.
* **Three distinct biological groupings emerge:**
  * **Normal tissue cluster (bottom-right, green):** Solid Tissue Normal biopsies (`-11`, `PC1 ≈ 25 to 45`, `PC2 ≈ -25 to 0`) form a tight, well-separated cluster from almost all tumors.
  * **Main primary tumor cloud (center-left, blue):** Most Primary Tumors (`-01`, `PC1 ≈ -35 to 20`, `PC2 ≈ -20 to 15`) occupy a broad central-left region, with the 7 Metastatic samples (`-06`, red) embedded within the tumor space rather than forming technical outliers.
  * **Upper tumor sub-cluster (top-center, blue):** A distinct lobe of primary tumors rises sharply along positive PC2 (`PC2 ≈ 15 to 45`), pointing to a major intrinsic breast cancer subtype split that we can verify via PC gene loadings.

## Biological Drivers of PC1 and PC2
Biological Interpretation of PC Axes

In [59]:
# Extract gene loadings (weights) for PC1 and PC2 to identify the exact
# biological pathways driving the two primary axes of cohort variation
loadings = pd.DataFrame(
    pca.components_[:2].T,
    index=X_hvg.index,
    columns=["PC1", "PC2"]
)

# Map Ensembl IDs to HGNC symbols for biological interpretability, falling back
# to Ensembl IDs as a Series (pandas fillna does not accept raw Index objects)
mapped_genes = expr.loc[loadings.index, "gene"]
loadings["symbol"] = mapped_genes.fillna(loadings.index.to_series())

# Select the 8 strongest negative and 8 strongest positive gene weights per axis
# to contrast opposing transcriptional programs (e.g., tumor vs. normal, subtype markers)
top_pc1 = pd.concat([loadings.nsmallest(8, "PC1"), loadings.nlargest(8, "PC1")])
top_pc2 = pd.concat([loadings.nsmallest(8, "PC2"), loadings.nlargest(8, "PC2")])

fig = make_subplots(
    rows=1, cols=2,
    horizontal_spacing=0.16,
    subplot_titles=(
        "Top Gene Loadings Driving PC1 (14.7%)",
        "Top Gene Loadings Driving PC2 (12.9%)"
    )
)

# Panel 1: Negative weights pull biopsies toward Tumor (left); positive weights pull toward Normal (right)
fig.add_trace(
    go.Bar(
        x=top_pc1["PC1"],
        y=top_pc1["symbol"],
        orientation="h",
        marker_color=["#1f77b4" if v < 0 else "#2ca02c" for v in top_pc1["PC1"]],
        showlegend=False
    ),
    row=1, col=1
)

# Panel 2: Negative weights pull toward the main tumor cloud; positive weights pull toward the upper tumor lobe
fig.add_trace(
    go.Bar(
        x=top_pc2["PC2"],
        y=top_pc2["symbol"],
        orientation="h",
        marker_color=["#636efa" if v < 0 else "#d62728" for v in top_pc2["PC2"]],
        showlegend=False
    ),
    row=1, col=2
)

fig.update_layout(
    height=480,
    margin=dict(t=90, b=50, l=90, r=30),
    title_text="Biological Drivers of PC1 and PC2 (Top Positive & Negative Gene Loadings)"
)
fig.update_xaxes(title_text="PC1 Loading Weight", row=1, col=1)
fig.update_xaxes(title_text="PC2 Loading Weight", row=1, col=2)
fig.show()

* **PC1 (14.7%) separates normal mammary architecture from epithelial tumor cells:**
  * **Positive weights (green, pulling right toward Normal `-11`):** Dominated by normal myoepithelial, stromal, and tumor-suppressor markers such as **DMD**, **SFRP1**, **ANXA1**, **CAV2**, and **CRYAB** (`~0.047` to `0.050`).
    * DMD : 
    * SFRP1 : 
    * AXA1 : 
    * CAV2 : 
    * CRYAB : 
  * **Negative weights (blue, pulling left toward Primary Tumor `-01`):** Driven by epithelial and luminal breast cancer genes including **COX6C**, **SLC9A3R1**, **STARD10**, and **BCAS4** (`~-0.036` to `-0.040`).
    * COX6C : 
    * SLC9A3R1 : 
    * STARD10 : 
    * BCAS4 : 
      
        
### To be justified by gene analysis : 
* **PC2 (12.9%) captures the major Luminal vs. Proliferative/Basal tumor subtype split:** 
  * **Positive weights (red, pulling up into the upper tumor lobe):** Every top contributor is a cell-cycle and mitotic division gene—**CCNE1**, **CDC20**, **AURKB**, **CDCA8**, **KIF2C**, **NDC80**, **MCM10**, and **CDCA3** (`~0.047` to `0.051`)—identifying rapidly proliferating, high-grade tumors (typically Basal-like / triple-negative).
  * **Negative weights (purple, pulling down into the main tumor cloud):** Driven by hormone-receptor and luminal differentiation markers—**ERBB4**, **AR**, **SCUBE2**, **IL6ST**, **CCDC170**, and **TBC1D9** (`~-0.046` to `-0.050`)—confirming that the main tumor cloud corresponds to estrogen/androgen-regulated Luminal tumors.

## PCA Colored by Hospital Tissue Source Site (TSS)

In [60]:
# Quantify total distinct collection centers and display the largest contributors
# to assess institutional concentration across the cohort
tss_counts = pca_df["tss"].value_counts()
print(f"Total unique Tissue Source Sites (TSS): {len(tss_counts)}")
print("Top 6 hospital Tissue Source Sites (sample counts):")
print(tss_counts.head(6))

# Group smaller centers into 'Other TSS' so the legend remains legible while
# isolating the 6 major hospitals that account for over half of all biopsies
top_tss = tss_counts.head(6).index
pca_df["tss_group"] = pca_df["tss"].where(pca_df["tss"].isin(top_tss), "Other TSS")

fig = go.Figure()

# Plot 'Other TSS' first in muted grey so minor centers provide cohort context
# without visually obscuring potential clustering among major collection sites
other_sub = pca_df[pca_df["tss_group"] == "Other TSS"]
fig.add_trace(
    go.Scatter(
        x=other_sub["PC1"],
        y=other_sub["PC2"],
        mode="markers",
        name=f"Other TSS ({len(other_sub)})",
        text=other_sub["sample"] + " | " + other_sub["tissue"] + " | TSS: " + other_sub["tss"],
        hoverinfo="text",
        marker=dict(color="#cccccc", size=5, opacity=0.45)
    )
)

# Overlay the top 6 hospitals with distinct colors to check whether samples
# mix across biological subtypes or segregate into institution-specific batch clusters
tss_palette = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd", "#8c564b"]
for tss_code, color in zip(top_tss, tss_palette):
    sub = pca_df[pca_df["tss"] == tss_code]
    fig.add_trace(
        go.Scatter(
            x=sub["PC1"],
            y=sub["PC2"],
            mode="markers",
            name=f"{tss_code} ({len(sub)})",
            text=sub["sample"] + " | " + sub["tissue"] + " | TSS: " + sub["tss"],
            hoverinfo="text",
            marker=dict(color=color, size=6, opacity=0.75)
        )
    )

fig.update_layout(
    title=f"PCA (Top {top_n:,} HVGs) Colored by Hospital Tissue Source Site (TSS)",
    xaxis_title=f"PC1 ({pca.explained_variance_ratio_[0]:.1%})",
    yaxis_title=f"PC2 ({pca.explained_variance_ratio_[1]:.1%})",
    height=480,
    legend=dict(title="Collection Center (n)")
)
fig.show()

Total unique Tissue Source Sites (TSS): 40
Top 6 hospital Tissue Source Sites (sample counts):
tss
BH    227
E2    104
A2    102
A8     84
D8     78
E9     77
Name: count, dtype: int64


* **No hospital-driven batch clusters across tumor subtypes:** Across both the main luminal tumor cloud (center-left) and the upper proliferative tumor lobe (top-center), all six major collection centers (**BH**: 227, **E2**: 104, **A2**: 102, **A8**: 84, **D8**: 78, **E9**: 77; 54.8% of the cohort) and **Other TSS** (554 biopsies) intermingle thoroughly rather than segregating by hospital. This confirms that leading PCs capture tumor biology rather than institution-specific processing artifacts.
* **Institutional enrichment in the Normal cluster reflects paired-sampling protocols:** The bottom-right Solid Tissue Normal cluster (`PC1 ≈ 25 to 45`) contains a noticeably higher share of **BH** samples (blue) alongside **E9** and **A2**, whereas **A8** and **D8** are largely absent. This pattern arises because specific centers (notably `BH`) systematically collected adjacent normal tissue (`-11`) paired with primary tumors, while other hospitals submitted primary tumors (`-01`) only.
* **No batch-correction preprocessing required:** Because major variance axes are governed by tissue and subtype biology rather than technical center effects, applying batch-correction algorithms (such as ComBat) is unnecessary and avoids inadvertently removing biological signal tied to institutional tumor/normal ratios.

In [ ]:
 # TODO : audit clinical/survival tables to choose prediction target